In [ ]:
%%configure -f
{"vCores": 16, "defaultLakehouse": {"name": "diagnostic", "id": "9d10bce5-1edc-4875-83c4-ac0a98a02775", "workspaceId": "82ad2591-974a-4ad4-ace6-e24879274a4b"}}


# fabric-rlm document-redaction direct-vs-RLM evaluation

Fabric-native adaptation of `Trampoline-AI/predict-rlm/examples/document_redaction`.
Downloads `PNFS-Employment-Agreement-2025.pdf` (6 pages, dense PII) and asks the
model to identify ALL personally-identifiable information targets that should be
redacted. Compares **direct single-call** vs **fabric_rlm.RLM** coverage against
a ground-truth list of PII strings mined directly from the input PDF.

| Mode | Trigger | LM | Output root |
|---|---|---|---|
| **Fabric** | `/lakehouse/default` exists | `FabricChatLM` (notebook identity, no key) | `/lakehouse/default/Files/fabric_rlm_document_redaction/<run_id>/` |
| **Local** | otherwise | `fabric_rlm.OpenAILM` (`OPENAI_API_KEY`) | `./_local_runs/document_redaction/<run_id>/` |

Scoring rewards each known PII string identified as a redaction target plus
each PII category covered. Only the structured `targets` list is scored — we
do NOT actually apply the redactions to a PDF (the RedactionResult schema is
sufficient for evaluation purposes).


In [ ]:
DR_RUN_ID = ''
DR_MODEL = 'gpt-5'
DR_MAX_TURNS = 20
DR_DIRECT_TEXT_CHARS = 60000
DR_PDF_URL = 'https://raw.githubusercontent.com/Trampoline-AI/predict-rlm/2d93675d6d69b45f9eda9b8fc01e178323f8e6cb/examples/document_redaction/sample/input/PNFS-Employment-Agreement-2025.pdf'
DR_TIMEOUT_SECONDS = 1500


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys, time, hashlib, shutil, traceback, uuid

LAKEHOUSE_ROOT = Path('/lakehouse/default')
FABRIC_RUNTIME = LAKEHOUSE_ROOT.exists() and (LAKEHOUSE_ROOT / 'Files').exists()

RUN_ID = str(globals().get('DR_RUN_ID') or '').strip() or (
    time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:8]
)

if FABRIC_RUNTIME:
    FILES_ROOT = LAKEHOUSE_ROOT / 'Files'
    RUN_ROOT = FILES_ROOT / 'fabric_rlm_document_redaction' / RUN_ID
else:
    FILES_ROOT = Path.cwd() / '_local_runs'
    RUN_ROOT = FILES_ROOT / 'document_redaction' / RUN_ID

INPUT_ROOT = RUN_ROOT / 'input'
DIRECT_ROOT = RUN_ROOT / 'direct'
RLM_ROOT = RUN_ROOT / 'rlm'
TRAJECTORY_ROOT = RUN_ROOT / 'trajectories'
for folder in [RUN_ROOT, INPUT_ROOT, DIRECT_ROOT, RLM_ROOT, TRAJECTORY_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

STAGE_EVENTS_PATH = RUN_ROOT / 'stage_events.jsonl'

def write_stage(stage, **details):
    payload = {'ts': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'stage': stage, **details}
    with open(STAGE_EVENTS_PATH, 'a', encoding='utf-8') as handle:
        handle.write(json.dumps(payload, ensure_ascii=False, default=str) + '\n')
        handle.flush()
    print(f'STAGE {stage}: {details}')

def write_json(relative_path, payload):
    path = RUN_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    return path

write_stage('bootstrap_start', run_root=str(RUN_ROOT), runtime='fabric' if FABRIC_RUNTIME else 'local')
print(f'Redaction run root: {RUN_ROOT}')

EXPECTED_SKILLS = ['pdf_document_analysis', 'validation', 'error_handling']

def _imports_ok():
    try:
        import fabric_rlm  # noqa: F401
        from fabric_rlm import File, RLM  # noqa: F401
        import fitz  # noqa: F401
        import nest_asyncio  # noqa: F401
        return True
    except Exception as exc:
        write_stage('import_probe_failed', error=repr(exc))
        return False

if FABRIC_RUNTIME:
    LOCAL_WHEEL_PATH = Path(os.environ.get('FABRIC_RLM_WHEEL', str(FILES_ROOT / 'fabric_rlm_longcot' / 'wheels' / 'fabric_rlm-0.1.8-py3-none-any.whl')))
    ISOLATED_DEPS_TARGET = Path(os.environ.get('FABRIC_RLM_DR_DEPS_TARGET', str(FILES_ROOT / 'fabric_rlm_document_redaction' / '_deps' / (LOCAL_WHEEL_PATH.stem + '-dr'))))

    def _prepend_path(p):
        s = str(p)
        if s not in sys.path:
            sys.path.insert(0, s)
        existing = os.environ.get('PYTHONPATH', '')
        if not existing or existing.split(os.pathsep)[0] != s:
            os.environ['PYTHONPATH'] = s + (os.pathsep + existing if existing else '')

    def _clear_modules(prefixes):
        for name in list(sys.modules):
            if any(name == p or name.startswith(p + '.') for p in prefixes):
                del sys.modules[name]

    if not ISOLATED_DEPS_TARGET.exists() and LOCAL_WHEEL_PATH.exists():
        ISOLATED_DEPS_TARGET.mkdir(parents=True, exist_ok=True)
        write_stage('bootstrap_install_start', wheel=str(LOCAL_WHEEL_PATH), target=str(ISOLATED_DEPS_TARGET))
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall',
                               '--no-deps', '--target', str(ISOLATED_DEPS_TARGET), str(LOCAL_WHEEL_PATH)])
    if ISOLATED_DEPS_TARGET.exists():
        _prepend_path(ISOLATED_DEPS_TARGET)
        _clear_modules(['fabric_rlm'])

    if not _imports_ok():
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
                               '--target', str(ISOLATED_DEPS_TARGET), 'pymupdf>=1.24.0,<1.25', 'nest_asyncio>=1.6'])
        _prepend_path(ISOLATED_DEPS_TARGET)
        _clear_modules(['fitz', 'pymupdf'])

    if not _imports_ok():
        raise ImportError('fabric_rlm/pymupdf unavailable after Fabric bootstrap')
else:
    if not _imports_ok():
        print('Local imports missing - attempting `pip install -e . pymupdf` ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(Path.cwd()), 'pymupdf>=1.24.0', 'nest_asyncio>=1.6'])
        for k in list(sys.modules):
            if k == 'fabric_rlm' or k.startswith('fabric_rlm.') or k in {'fitz', 'pymupdf'}:
                del sys.modules[k]
        if not _imports_ok():
            raise ImportError('fabric_rlm/pymupdf unavailable after local bootstrap')

import fabric_rlm
available = set(fabric_rlm.list_skills())
missing = set(EXPECTED_SKILLS) - available
if missing:
    raise RuntimeError(f'fabric_rlm at {fabric_rlm.__file__} missing skills: {sorted(missing)}; available={sorted(available)}')
write_stage('bootstrap_ok', fabric_rlm=str(fabric_rlm.__file__), version=getattr(fabric_rlm, '__version__', '?'),
            available_skills=sorted(available), runtime='fabric' if FABRIC_RUNTIME else 'local')


In [ ]:
import re, urllib.error, urllib.request
import fitz
import fabric_rlm
from fabric_rlm import File, RLM

MODEL = str(globals().get('DR_MODEL') or 'gpt-5')
MAX_TURNS = int(globals().get('DR_MAX_TURNS') or 20)
DIRECT_TEXT_CHARS = int(globals().get('DR_DIRECT_TEXT_CHARS') or 60000)
PDF_URL = str(globals().get('DR_PDF_URL') or '').strip()
TIMEOUT_SECONDS = int(globals().get('DR_TIMEOUT_SECONDS') or 1500)
MODEL_KWARGS = {'max_tokens': 16000, 'temperature': 1.0}
_model_short = MODEL.split('/')[-1].lower()
_is_reasoning = _model_short.startswith(('gpt-5', 'o1', 'o3', 'o4')) and not _model_short.startswith('gpt-5-chat')
if _is_reasoning:
    MODEL_KWARGS = {'max_tokens': 32000, 'reasoning_effort': 'low'}

if not PDF_URL:
    raise ValueError('DR_PDF_URL must be set.')

OUTPUT_SCHEMA_HINT = '''
Return an object with these keys:
  - total_redactions (int, count of distinct PII target strings identified)
  - targets (list of objects, each with keys:
      text (str, the EXACT verbatim PII string as it appears in the PDF),
      category (str, one of: person_name, sin, social_insurance_number,
                phone, email, address, account_number, date_of_birth,
                institution_number, transit_number),
      page (int, 1-based page number where it appears),
      reason (str, brief justification for redaction))
  - page_summaries (list of {page: int, redaction_count: int})
  - summary (str, one-paragraph overview)
Identify EVERY occurrence — including duplicates on different pages and
BOTH the employee and the beneficiary/spouse PII. Do NOT redact
employer/business identifiers (PNFS, the company address, branch numbers).
'''.strip()

CRITERIA = '''
Identify ALL personally-identifiable information (PII) in an employment
agreement that should be redacted to protect both the employee and any third
parties (spouse/beneficiary, emergency contacts).

1. Survey the document - page count, sections (parties, compensation,
   benefits, signatures).
2. For EACH PII occurrence (not unique strings - every occurrence on every
   page) identify: the verbatim text, its category, and which page.
3. Cover ALL PII categories present:
     - person_name (employee, spouse/beneficiary, signatories,
       reporting manager, emergency contact)
     - sin / social_insurance_number (Canadian SIN, format NNN-NNN-NNN)
     - phone (mobile, emergency contact, work)
     - email (personal and any work)
     - address (home address - street, city, postal code together)
     - account_number (bank account, RRSP account, employee ID)
     - date_of_birth
     - institution_number / transit_number (banking)
4. EXCLUDE the employer's business identifiers (company name PNFS,
   company address, business phone) - those are not personal PII.
'''.strip()

# Ground-truth PII strings mined from PNFS-Employment-Agreement-2025.pdf.
# Each string here is one scoring anchor; a hit means the model included
# that exact substring in any target.text. Categories are validated
# separately (model must include each ground-truth category at least once).
GROUND_TRUTH = {
    'person_names': [
        'Margaret Elisabeth Thornbury-Watson',  # employee, multiple occurrences
        'David Watson',                         # spouse / emergency contact / beneficiary
        'Robert Chen',                          # employer rep signatory
        'James Harrington',                     # reporting manager
    ],
    'sins': [
        '847-291-036',  # employee SIN
        '912-347-058',  # beneficiary SIN
    ],
    'phones': [
        '(604) 889-3247',  # employee mobile
        '(604) 773-5518',  # emergency contact / spouse
    ],
    'emails': [
        'm.thornbury.watson@gmail.com',
    ],
    'addresses': [
        '2934 Cypress Crescent',  # home address (partial substring is fine)
    ],
    'account_numbers': [
        '5127849',  # bank account number
    ],
    'dates_of_birth': [
        'September 12, 1987',  # employee DOB
        'June 3, 1985',        # beneficiary DOB
    ],
    # Distinct categories the model must reference at least once.
    'categories_required': [
        'person_name', 'sin', 'phone', 'email', 'address',
        'account_number', 'date_of_birth',
    ],
    # Total scoring anchors:
    #   13 PII string anchors (4 names + 2 SIN + 2 phones + 1 email + 1 address + 1 account + 2 DOB)
    # + 7 category anchors
    # + 1 minimum-target-count anchor (>= 13 total targets)
    # = 21 anchors
}

if FABRIC_RUNTIME:
    class FabricChatLM:
        def __init__(self, model, timeout=360, **default_kwargs):
            from synapse.ml.fabric.service_discovery import get_fabric_env_config
            from synapse.ml.fabric.token_utils import TokenUtils
            env = get_fabric_env_config().fabric_env_config
            base = f'{env.ml_workload_endpoint}cognitive/openai'.rstrip('/')
            self.model = model
            self.timeout = timeout
            self.default_kwargs = dict(default_kwargs)
            self.headers = {'Authorization': TokenUtils().get_openai_auth_header(), 'Content-Type': 'application/json'}
            self.urls = [
                f'{base}/openai/deployments/{model}/chat/completions?api-version=2025-04-01-preview',
                f'{base}/deployments/{model}/chat/completions?api-version=2025-04-01-preview',
            ]
        def __call__(self, *, messages, **kwargs):
            ck = {**self.default_kwargs, **kwargs}
            body = {'messages': messages}
            if ck.get('temperature') is not None:
                body['temperature'] = ck['temperature']
            if ck.get('max_tokens') is not None:
                body['max_completion_tokens'] = int(ck['max_tokens'])
            data = json.dumps(body).encode('utf-8')
            errors = []
            for i, url in enumerate(self.urls):
                req = urllib.request.Request(url, data=data, headers=self.headers, method='POST')
                try:
                    with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                        payload = json.loads(resp.read().decode('utf-8'))
                    choice = (payload.get('choices') or [{}])[0]
                    msg = choice.get('message') or {}
                    return {'content': msg.get('content') or choice.get('text') or '',
                            'usage': payload.get('usage') or {}, 'model': payload.get('model') or self.model}
                except urllib.error.HTTPError as exc:
                    detail = exc.read().decode('utf-8', errors='replace')[:2000]
                    errors.append({'url_index': i, 'status': exc.code, 'detail': detail})
                    if exc.code not in {400, 404}:
                        break
                except Exception as exc:
                    errors.append({'url_index': i, 'error': repr(exc)})
                    break
            raise RuntimeError(f'FabricChatLM call failed: {errors}')

    def make_direct_lm():
        return FabricChatLM(MODEL, timeout=TIMEOUT_SECONDS, **MODEL_KWARGS)
    def make_rlm_lm():
        return FabricChatLM(MODEL, timeout=TIMEOUT_SECONDS, **MODEL_KWARGS)
    SUB_LM_SPEC = 'fabric/' + MODEL
else:
    def _detect_local_provider():
        if os.environ.get('OPENAI_API_KEY'): return 'openai'
        if os.environ.get('OPENROUTER_API_KEY'): return 'openrouter'
        raise RuntimeError('Local runtime requires OPENAI_API_KEY or OPENROUTER_API_KEY.')
    def _local_model_spec():
        provider = _detect_local_provider()
        if provider == 'openai': return MODEL
        if MODEL.startswith('openrouter/'): return MODEL
        if '/' in MODEL: return 'openrouter/' + MODEL
        return 'openrouter/openai/' + MODEL
    def _make_local_lm(**extra_kwargs):
        import dspy
        provider = _detect_local_provider()
        kwargs = dict(MODEL_KWARGS); kwargs.update(extra_kwargs)
        if provider == 'openai':
            from fabric_rlm import OpenAILM
            return OpenAILM(MODEL, **kwargs)
        return dspy.LM(model=_local_model_spec(),
                       api_key=os.environ['OPENROUTER_API_KEY'],
                       api_base='https://openrouter.ai/api/v1', **kwargs)
    class _LocalDirectLM:
        def __init__(self, timeout=360, **default_kwargs):
            self.model = _local_model_spec(); self.timeout = timeout
            self._lm = _make_local_lm(**default_kwargs)
        def __call__(self, *, messages, **_kwargs):
            out = self._lm(messages=messages)
            if isinstance(out, list) and out:
                first = out[0]
                if isinstance(first, str): return {'content': first, 'usage': {}, 'model': self.model}
                if isinstance(first, dict): return {'content': first.get('content', ''), 'usage': {}, 'model': self.model}
            if isinstance(out, str): return {'content': out, 'usage': {}, 'model': self.model}
            return {'content': str(out), 'usage': {}, 'model': self.model}
    def make_direct_lm():
        _detect_local_provider(); return _LocalDirectLM(timeout=TIMEOUT_SECONDS, **MODEL_KWARGS)
    def make_rlm_lm():
        return _make_local_lm()
    try: SUB_LM_SPEC = _local_model_spec()
    except Exception: SUB_LM_SPEC = ('openai/' + MODEL) if '/' not in MODEL else MODEL

def response_to_text(response):
    if isinstance(response, str): return response
    if isinstance(response, dict): return str(response.get('content') or response.get('text') or response)
    return str(getattr(response, 'content', response))

def extract_json_object(text):
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    try:
        return json.loads(cleaned)
    except Exception:
        start = cleaned.find('{'); end = cleaned.rfind('}')
        if start >= 0 and end > start:
            return json.loads(cleaned[start:end + 1])
        raise

def normalize_items(value):
    if value is None: return []
    if isinstance(value, list): return value
    return [value]

def _norm_text(s):
    return re.sub(r'\s+', ' ', str(s or '')).strip().lower()

def _norm_category(s):
    c = _norm_text(s).replace('-', '_').replace(' ', '_')
    # collapse synonyms
    if c in {'sin', 'social_insurance_number', 'social_insurance', 'sin_number'}:
        return 'sin'
    if c in {'phone', 'phone_number', 'mobile', 'mobile_phone', 'telephone', 'cell', 'cell_phone'}:
        return 'phone'
    if c in {'email', 'email_address', 'e_mail'}:
        return 'email'
    if c in {'name', 'person_name', 'full_name', 'employee_name', 'beneficiary_name', 'signatory_name'}:
        return 'person_name'
    if c in {'address', 'home_address', 'street_address', 'mailing_address', 'residential_address'}:
        return 'address'
    if c in {'account_number', 'bank_account', 'bank_account_number', 'account', 'rrsp_account', 'rrsp_account_number'}:
        return 'account_number'
    if c in {'dob', 'date_of_birth', 'birth_date', 'birthdate'}:
        return 'date_of_birth'
    return c

def score_redaction(analysis):
    '''Score the structured redaction targets against ground-truth PII.

    Anchors:
      - 13 PII string anchors (each ground-truth string must appear as a
        substring in some target.text, normalized whitespace + lowercase).
      - 7 category anchors (each required category must be referenced by
        at least one target).
      - 1 minimum-count anchor (total >= 13 distinct targets).
    '''
    targets = normalize_items(analysis.get('targets'))
    target_texts_norm = [_norm_text(t.get('text') if isinstance(t, dict) else t)
                         for t in targets]
    target_cats_norm = set(_norm_category(t.get('category') if isinstance(t, dict) else '')
                           for t in targets)
    # Concatenated for substring search.
    target_blob = ' || '.join(target_texts_norm)

    groups = {}
    total_hits = 0
    total_possible = 0
    details = {}

    for group_name, items in [
        ('person_names', GROUND_TRUTH['person_names']),
        ('sins', GROUND_TRUTH['sins']),
        ('phones', GROUND_TRUTH['phones']),
        ('emails', GROUND_TRUTH['emails']),
        ('addresses', GROUND_TRUTH['addresses']),
        ('account_numbers', GROUND_TRUTH['account_numbers']),
        ('dates_of_birth', GROUND_TRUTH['dates_of_birth']),
    ]:
        hits = []
        misses = []
        for item in items:
            needle = _norm_text(item)
            if needle and needle in target_blob:
                hits.append(item)
            else:
                misses.append(item)
        groups[group_name] = {'hits': hits, 'misses': misses,
                              'score': len(hits), 'possible': len(items)}
        total_hits += len(hits)
        total_possible += len(items)

    # Category coverage
    cat_hits = []
    cat_misses = []
    for cat in GROUND_TRUTH['categories_required']:
        if cat in target_cats_norm:
            cat_hits.append(cat)
        else:
            cat_misses.append(cat)
    groups['categories'] = {'hits': cat_hits, 'misses': cat_misses,
                            'score': len(cat_hits), 'possible': len(GROUND_TRUTH['categories_required'])}
    total_hits += len(cat_hits)
    total_possible += len(GROUND_TRUTH['categories_required'])

    # Minimum target count anchor
    min_target_count = 13
    count_hit = len(targets) >= min_target_count
    groups['min_target_count'] = {
        'score': 1 if count_hit else 0, 'possible': 1,
        'actual': len(targets), 'expected_min': min_target_count,
    }
    if count_hit:
        total_hits += 1
    total_possible += 1

    return {
        'score': total_hits, 'possible': total_possible,
        'coverage': total_hits / max(total_possible, 1),
        'groups': groups,
        'target_count_actual': len(targets),
        'distinct_categories_actual': sorted(target_cats_norm),
        'total_redactions_reported': analysis.get('total_redactions'),
    }


In [ ]:
pdf_paths = []
name = PDF_URL.rsplit('/', 1)[-1]
target = INPUT_ROOT / name
if not target.exists():
    write_stage('download_pdf_start', url=PDF_URL, path=str(target))
    req = urllib.request.Request(PDF_URL, headers={'User-Agent': 'fabric-rlm-document-redaction'})
    with urllib.request.urlopen(req, timeout=180) as resp:
        target.write_bytes(resp.read())
    write_stage('download_pdf_done', bytes=target.stat().st_size, path=str(target))
else:
    write_stage('download_pdf_cached', path=str(target), bytes=target.stat().st_size)
pdf_paths.append(target)

pdf_text_blocks = []
pdf_metadata = []
for p in pdf_paths:
    doc = fitz.open(p)
    pages = [{'page': i + 1, 'text': page.get_text('text')} for i, page in enumerate(doc)]
    doc.close()
    full_text = '\n\n'.join(f'--- Page {row["page"]} ---\n{row["text"]}' for row in pages)
    text_path = INPUT_ROOT / (p.stem + '.txt')
    text_path.write_text(full_text, encoding='utf-8')
    excerpt = full_text[:DIRECT_TEXT_CHARS]
    pdf_text_blocks.append((p.name, excerpt))
    pdf_metadata.append({
        'name': p.name, 'pdf_path': str(p), 'text_path': str(text_path),
        'page_count': len(pages), 'text_chars': len(full_text),
        'direct_excerpt_chars': len(excerpt),
    })

manifest = {
    'run_id': RUN_ID, 'runtime': 'fabric' if FABRIC_RUNTIME else 'local',
    'source': 'Trampoline-AI/predict-rlm examples/document_redaction sample PDF',
    'pdf_url': PDF_URL,
    'documents': pdf_metadata,
    'model': MODEL, 'max_turns': MAX_TURNS,
    'fabric_rlm_version': getattr(fabric_rlm, '__version__', 'unknown'),
    'python': sys.version, 'platform': platform.platform(),
}
write_json('manifest.json', manifest)
write_stage('documents_ready',
            docs=[{'name': m['name'], 'pages': m['page_count'], 'text_chars': m['text_chars']}
                  for m in pdf_metadata])


In [ ]:
direct_lm = make_direct_lm()
direct_user_parts = [CRITERIA, OUTPUT_SCHEMA_HINT,
                     'You are receiving the extracted text of a single employment-agreement PDF. '
                     'This is a single-pass direct baseline; use only this text.']
for name, excerpt in pdf_text_blocks:
    direct_user_parts.append(f'\n=== DOCUMENT: {name} ===\n{excerpt}')
direct_messages = [
    {'role': 'system',
     'content': ('You identify ALL personally-identifiable information (PII) targets in '
                 'an employment-agreement PDF and return ONLY valid JSON with keys '
                 'total_redactions, targets, page_summaries, summary. Cover every '
                 'occurrence of every PII string on every page where it appears.')},
    {'role': 'user', 'content': '\n\n'.join(direct_user_parts)},
]
write_stage('direct_start', model=MODEL,
            total_text_chars=sum(len(e) for _, e in pdf_text_blocks))
direct_started = time.perf_counter()
direct_response = direct_lm(messages=direct_messages)
direct_duration = time.perf_counter() - direct_started
direct_text = response_to_text(direct_response)
(DIRECT_ROOT / 'raw_response.txt').write_text(direct_text, encoding='utf-8')
try:
    direct_analysis = extract_json_object(direct_text)
    direct_error = None
except Exception as exc:
    direct_analysis = {'total_redactions': 0, 'targets': [], 'page_summaries': [],
                       'summary': direct_text[:500]}
    direct_error = repr(exc)
direct_score = score_redaction(direct_analysis)
write_json('direct/analysis.json', direct_analysis)
write_json('direct/score.json', direct_score)
write_stage('direct_done', duration_s=direct_duration,
            score=direct_score['score'], possible=direct_score['possible'],
            coverage=round(direct_score['coverage'], 3),
            target_count=direct_score['target_count_actual'],
            parse_error=direct_error)
print(f"DIRECT  : score={direct_score['score']}/{direct_score['possible']}  "
      f"coverage={direct_score['coverage']:.0%}  "
      f"targets={direct_score['target_count_actual']}  "
      f"in {direct_duration:.1f}s")


In [ ]:
RLM_TASK = '''
Identify ALL personally-identifiable information (PII) redaction targets in
the provided employment-agreement PDF.

You have access to the preloaded `pdf_document_analysis` skill - follow it for PDF work:
- Use Python and PyMuPDF (`fitz`) to open the PDF and record `page_count`.
- For each page, render an image at ~200 DPI to a data URI and use `predict()`
  to enumerate every PII occurrence with category + page.
- Cross-check using raw PDF text (`page.get_text("text")`) - text extraction
  is more reliable than vision for SINs, phone numbers, account numbers, and
  emails. Use the text as ground truth for the EXACT verbatim string.
- Use `asyncio.gather()` over independent `await predict(...)` calls per page.

Categories to identify (use these category labels exactly):
  person_name, sin, phone, email, address, account_number, date_of_birth,
  institution_number, transit_number

Include the employee, the spouse/beneficiary, the emergency contact, the
reporting manager, and ALL signatories. EXCLUDE the employer's business
identifiers (company name, business address, business phone).

Write a JSON analysis to {output_dir}/analysis.json with the full
targets list.

Before SUBMIT, run a self-check and repair any failure:
- Every page is represented (page_summaries covers pages 1..N).
- Each Canadian SIN matches the format NNN-NNN-NNN.
- Each phone matches a North-American phone format.
- Each email contains an `@`.
- For each PII string, EVERY occurrence on EVERY page is listed (do not
  deduplicate across pages).
- total_redactions = len(targets).

Call SUBMIT(total_redactions=..., targets=..., page_summaries=...,
            summary=..., analysis_path=...).
Required JSON-friendly shapes:
- targets: list of dicts with keys text, category, page, reason.
- page_summaries: list of dicts with keys page, redaction_count.
'''.strip()

write_stage('rlm_setup_start', model=MODEL, max_turns=MAX_TURNS)
try:
    rlm_lm = make_rlm_lm()
    doc_files = [File(str(p)) for p in pdf_paths]
    rlm = RLM.from_task(
        task=RLM_TASK,
        inputs={'document': doc_files[0], 'criteria': CRITERIA, 'output_dir': str(RLM_ROOT)},
        outputs=['total_redactions', 'targets', 'page_summaries', 'summary', 'analysis_path'],
        lm=rlm_lm,
        sub_lm=SUB_LM_SPEC,
        max_turns=MAX_TURNS,
        skills=['pdf_document_analysis'],
        enable_skill_autoloading=True,
        timeout=TIMEOUT_SECONDS,
    )
except Exception as exc:
    tb = traceback.format_exc()
    write_stage('rlm_setup_failed', error=repr(exc), traceback=tb[-4000:])
    (RLM_ROOT / 'setup_error.txt').write_text(tb, encoding='utf-8')
    raise
write_stage('rlm_start', model=MODEL, max_turns=MAX_TURNS)
rlm_started = time.perf_counter()
try:
    rlm_result = rlm.run()
except Exception as exc:
    tb = traceback.format_exc()
    write_stage('rlm_run_failed', error=repr(exc), traceback=tb[-4000:],
                duration_s=time.perf_counter() - rlm_started)
    (RLM_ROOT / 'run_error.txt').write_text(tb, encoding='utf-8')
    raise
rlm_duration = time.perf_counter() - rlm_started

trajectory_path = TRAJECTORY_ROOT / 'document_redaction_rlm.jsonl'
rlm_result.trajectory.write_jsonl(trajectory_path)

if rlm_result.submitted and rlm_result.payload:
    rlm_analysis = dict(rlm_result.payload)
else:
    rlm_analysis = {'total_redactions': 0, 'targets': [], 'page_summaries': [],
                    'summary': '', 'failure_reason': rlm_result.failure_reason}

rlm_score = score_redaction(rlm_analysis)
write_json('rlm/analysis.json', rlm_analysis)
write_json('rlm/score.json', rlm_score)
write_stage('rlm_done', duration_s=rlm_duration, submitted=rlm_result.submitted,
            turns=len(rlm_result.trajectory.turns),
            score=rlm_score['score'], possible=rlm_score['possible'],
            coverage=round(rlm_score['coverage'], 3),
            target_count=rlm_score['target_count_actual'])
print(f"RLM     : submitted={rlm_result.submitted}  turns={len(rlm_result.trajectory.turns)}  "
      f"score={rlm_score['score']}/{rlm_score['possible']}  "
      f"coverage={rlm_score['coverage']:.0%}  "
      f"targets={rlm_score['target_count_actual']}  "
      f"in {rlm_duration:.1f}s")


In [ ]:
record = {
    'run_id': RUN_ID, 'runtime': 'fabric' if FABRIC_RUNTIME else 'local',
    'model': MODEL,
    'source_example': 'https://github.com/Trampoline-AI/predict-rlm/tree/main/examples/document_redaction',
    'documents': [m['name'] for m in pdf_metadata],
    'direct': {'duration_s': direct_duration, 'score': direct_score, 'parse_error': direct_error},
    'rlm':    {'duration_s': rlm_duration,    'submitted': rlm_result.submitted,
               'failure_reason': rlm_result.failure_reason, 'score': rlm_score,
               'turns': len(rlm_result.trajectory.turns),
               'trajectory_path': str(trajectory_path)},
}
record['rlm_upgrade'] = (
    rlm_score['score'] > direct_score['score']
    or rlm_score['coverage'] > direct_score['coverage']
)
write_json('record.json', record)

print()
print('=' * 70)
print('DOCUMENT REDACTION - DIRECT vs RLM')
print('=' * 70)
for label, sc, dur in [('DIRECT', direct_score, direct_duration),
                       ('RLM   ', rlm_score, rlm_duration)]:
    print(f'{label}:  coverage={sc["coverage"]:.0%}  '
          f'({sc["score"]}/{sc["possible"]} anchors)  '
          f'targets={sc["target_count_actual"]}  '
          f'in {dur:.1f}s')
print()
print('Per-anchor-group hit rate:')
for group in ['person_names', 'sins', 'phones', 'emails', 'addresses',
              'account_numbers', 'dates_of_birth', 'categories', 'min_target_count']:
    d = direct_score['groups'].get(group, {'score': 0, 'possible': 0})
    r = rlm_score['groups'].get(group, {'score': 0, 'possible': 0})
    print(f'  {group:22s}  direct {d["score"]}/{d["possible"]}   rlm {r["score"]}/{r["possible"]}')
print()
print(f'Run root: {RUN_ROOT}')
print(f'Trajectory: {trajectory_path}')
print(f'RLM upgraded over direct? {record["rlm_upgrade"]}')
